# 22. 21번과 같은 Kiwi BM25 단독 검색 + Qwen3.8 답변 비교

**21번과 동일한 문서 로딩, 전처리, 청킹, 사업 식별, Kiwi 토큰화, 사업별 BM25 검색**을 사용합니다. 같은 질문에서 검색 상위 20개를 먼저 보여주고, 그중 앞의 4개를 Qwen3.8-27B에 넣어 답변을 만듭니다. 21번에는 생성 답변이 없으므로 검색 순위의 일치 여부와 Qwen 답변을 함께 확인하는 비교입니다.

- 질문: 서로 다른 3개 사업에서 각 1문항. `PREVIEW_COUNT`를 바꿀 수 있습니다.
- 검색: 21번과 같이 **사업별 Kiwi BM25 상위 20개**. Dense·RRF·재순위화 없음.
- 생성: 검색 상위 4개를 사용. 무료 Colab T4의 문맥 길이를 고려한 설정입니다.
- 기존 21번 결과 `MyDrive/bm25_only_comparison_30.json`이 있으면 문서·청크 내용과 Kiwi 검색 순위가 같은지 자동 확인합니다. 없으면 이 노트북 안에서 단독 검색 결과와 Qwen 답변을 나란히 볼 수 있습니다.
- 생성 모델: [`Qwen/Qwen3.8-27B`](https://huggingface.co/Qwen/Qwen3.8-27B)의 [Q3_K_S 양자화 파일](https://huggingface.co/bartowski/Qwen3.8-27B-GGUF). 양자화로 원본보다 품질이 낮을 수 있고, 아직 Colab T4에서 실제 실행을 확인하지 않았습니다.
- LangSmith는 맨 아래에 준비되어 있으며 기본값은 실행하지 않습니다.

Colab에서 무료 T4 GPU를 선택하고 위에서 아래로 실행하세요. Drive에 `MyDrive/data` 문서, `MyDrive/data_list.csv`, `MyDrive/rag_eval_dataset_3x10.json`이 필요합니다. 모델 다운로드와 빌드로 첫 실행은 오래 걸릴 수 있습니다.


In [ ]:
%pip install -q pandas pymupdf4llm langchain-hwp-hwpx-loader langchain-text-splitters langchain-community rank_bm25 kiwipiepy huggingface_hub openai langsmith


In [ ]:
from pathlib import Path
import os
import hashlib
import json
import re
import unicodedata

import pandas as pd
import pymupdf4llm
import torch
from google.colab import drive
from hwp_hwpx_parser import ExtractOptions, TableStyle
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_hwp_hwpx import HwpHwpxLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from kiwipiepy import Kiwi

drive.mount('/content/drive')
DATA_DIR = Path('/content/drive/MyDrive/data')
METADATA_CSV = Path('/content/drive/MyDrive/data_list.csv')
QUESTION_JSON = Path('/content/drive/MyDrive/rag_eval_dataset_3x10.json')

if not torch.cuda.is_available():
    raise RuntimeError('Colab 메뉴에서 런타임 > 런타임 유형 변경 > GPU를 선택하세요.')
print('GPU:', torch.cuda.get_device_name(0))


## 1. 문서 불러오기

21번과 같은 순서로 HWP/HWPX와 PDF를 읽고 CSV 메타데이터를 연결합니다.


In [ ]:
folder = str(DATA_DIR)
metadata_csv = str(METADATA_CSV)

# 문서 순서를 고정해 다시 실행해도 청크 번호가 같은 순서로 생성되도록 합니다.
sorted_file_names = sorted(
    os.listdir(folder),
    key=lambda name: (unicodedata.normalize("NFKC", name).casefold(), name),
)
print("정렬한 파일 수:", len(sorted_file_names))
def normalize_filename(name):
    name = str(name).rsplit(".", 1)[0]
    name = unicodedata.normalize("NFKC", name)
    return re.sub(r"\s+", " ", name).strip()

df = pd.read_csv(metadata_csv, encoding="utf-8-sig")
loaded_files = []
for name in sorted_file_names:
    full_path = os.path.join(folder, name)
    if not os.path.isfile(full_path):
        continue
    extension = name.rsplit(".", 1)[-1].lower()
    if extension == "pdf":
        text = pymupdf4llm.to_markdown(full_path)
    elif extension in ("hwp", "hwpx"):
        options = ExtractOptions(table_style=TableStyle.MARKDOWN)
        loader = HwpHwpxLoader(full_path, mode="single", extract_options=options)
        text = "\n\n".join(doc.page_content for doc in loader.load())
    else:
        continue
    if not text.strip():
        raise ValueError(f"본문이 비어 있습니다: {name}")
    loaded_files.append((name, text))
if not loaded_files:
    raise ValueError(f"PDF/HWP/HWPX 문서를 찾지 못했습니다: {folder}")
print("로드한 문서:", len(loaded_files))
metadata_columns = [
    "공고 번호", "공고 차수", "사업명", "사업 금액", "발주 기관",
    "공개 일자", "입찰 참여 시작일", "입찰 참여 마감일", "파일명",
]
missing_columns = set(metadata_columns) - set(df.columns)
if missing_columns:
    raise ValueError(f"data_list.csv에 없는 열: {sorted(missing_columns)}")
documents = []
unmatched_files = []
for name, text in loaded_files:
    result_df = df[df["파일명"].apply(normalize_filename) == normalize_filename(name)]
    if result_df.empty:
        unmatched_files.append(name)
        continue
    if len(result_df[["발주 기관", "사업명"]].drop_duplicates()) != 1:
        raise ValueError(f"파일의 사업 메타데이터가 여러 개입니다: {name}")
    row = result_df.iloc[0]
    metadata = {column: "" if pd.isna(row[column]) else str(row[column])
                for column in metadata_columns}
    documents.append(Document(page_content=text, metadata=metadata))
if not documents:
    raise ValueError("문서 파일명과 data_list.csv의 파일명을 확인하세요.")
print("메타데이터와 연결한 문서:", len(documents))
if unmatched_files:
    print("CSV와 연결되지 않아 제외한 파일:", unmatched_files)
for document in documents:
    text = document.page_content.replace("\r\n", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    document.page_content = re.sub(r"\n{3,}", "\n\n", text).strip()


## 2. 청킹과 Kiwi BM25 단독 검색

21번과 같은 700자 청크, 100자 겹침, 사업 식별, Kiwi 형태소 분석을 사용합니다. 상위 20개를 검색합니다.


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " "], chunk_size=700, chunk_overlap=100
)

chunks = splitter.split_documents(documents)
for chunk_id, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = chunk_id

print("청크 수:", len(chunks))

from collections import defaultdict
from langchain_community.retrievers import BM25Retriever


def normalize_project_text(value):
    return "".join(
        char for char in unicodedata.normalize("NFKC", str(value)).casefold()
        if char.isalnum()
    )


def project_title_matches_query(project_key, normalized_query):
    agency, title = project_key
    normalized_title = normalize_project_text(title)
    if normalized_title in normalized_query:
        return True

    # 기관명이 사업명 앞에 있지만 질문에서는 조사와 함께 분리된 경우
    normalized_agency = normalize_project_text(agency)
    if normalized_agency and normalized_title.startswith(normalized_agency):
        short_title = normalized_title[len(normalized_agency):]
        if len(short_title) >= 8 and normalized_agency in normalized_query:
            return short_title in normalized_query
    return False


def metadata_text(value):
    return "" if pd.isna(value) else str(value).strip()


# 줄임말을 쓰려면 아래에 "줄임말": ("발주 기관", "정식 사업명")을 추가합니다.
PROJECT_ALIASES = {}

project_chunks_by_key = defaultdict(list)
for doc in chunks:
    agency = metadata_text(doc.metadata.get("발주 기관"))
    title = metadata_text(doc.metadata.get("사업명"))
    if title:
        project_chunks_by_key[(agency, title)].append(doc)

project_catalog = sorted(project_chunks_by_key)
if not project_catalog:
    raise ValueError("사업명 메타데이터가 있는 청크가 없습니다.")
print("식별 가능한 사업:", len(project_catalog))


def choose_project(matches, query):
    matches = sorted(set(matches))
    if len(matches) == 1:
        return matches[0]

    normalized_query = normalize_project_text(query)
    agency_matches = [
        key for key in matches
        if key[0] and normalize_project_text(key[0]) in normalized_query
    ]
    if len(agency_matches) == 1:
        return agency_matches[0]

    labels = ", ".join(f'[{agency}] "{title}"' for agency, title in matches)
    raise ValueError(f"사업을 하나로 특정할 수 없습니다: {labels}")


def identify_project(query):
    """질문의 정식 사업명 또는 등록된 줄임말로 사업 하나를 고릅니다."""
    normalized_query = normalize_project_text(query)
    if not normalized_query:
        raise ValueError("질문이 비어 있습니다.")

    # 정식 사업명이 질문에 들어 있으면 포함 관계의 짧은 이름만 제외합니다.
    direct_matches = [
      key for key in project_catalog
      if project_title_matches_query(key, normalized_query)
    ]
    if direct_matches:
        specific_matches = [
            key for key in direct_matches
            if not any(
                normalize_project_text(key[1]) != normalize_project_text(other[1])
                and normalize_project_text(key[1]) in normalize_project_text(other[1])
                for other in direct_matches
            )
        ]
        return choose_project(specific_matches, query)

    # 정식 명칭이 없는 질문은 사용자가 등록한 줄임말만 사용합니다.
    alias_matches = [
        (alias, target) for alias, target in PROJECT_ALIASES.items()
        if normalize_project_text(alias) in normalized_query
    ]
    if alias_matches:
        longest = max(len(normalize_project_text(alias)) for alias, _ in alias_matches)
        targets = [
            target for alias, target in alias_matches
            if len(normalize_project_text(alias)) == longest
        ]
        unknown = [target for target in targets if target not in project_chunks_by_key]
        if unknown:
            raise ValueError(f"PROJECT_ALIASES의 사업이 문서 목록에 없습니다: {unknown}")
        return choose_project(targets, query)

    raise ValueError(
        "질문에서 사업명을 찾지 못했습니다. 정식 사업명을 질문에 적거나 "
        "PROJECT_ALIASES에 줄임말을 등록해 주세요."
    )



kiwi = Kiwi()
PUNCTUATION_TAGS = {"SF", "SP", "SS", "SE", "SO", "SW"}


def kiwi_morphs(text):
    """문서와 질문에 같은 Kiwi 형태소 분석을 적용합니다."""
    return [
        token.form
        for token in kiwi.tokenize(text)
        if token.form.strip() and token.tag not in PUNCTUATION_TAGS
    ]


SEARCH_TOP_K = 20
CONTEXT_TOP_K = 4
bm25_by_project = {}


def search_kiwi_bm25(question):
    # 21번처럼 해당 사업의 청크에서 Kiwi BM25 상위 20개를 찾습니다.
    project = identify_project(question)
    if len(project_chunks_by_key[project]) < SEARCH_TOP_K:
        raise ValueError(f'사업의 청크가 {SEARCH_TOP_K}개보다 적습니다: {project}')
    if project not in bm25_by_project:
        retriever = BM25Retriever.from_documents(
            project_chunks_by_key[project], preprocess_func=kiwi_morphs
        )
        retriever.k = SEARCH_TOP_K
        bm25_by_project[project] = retriever
    return project, bm25_by_project[project].invoke(question)


## 3. 21번 검색 결과와 비교

먼저 동일한 질문 3개에서 Kiwi BM25 단독 검색을 실행합니다. 21번 저장 JSON이 있으면 내용과 검색 순위가 같은지 확인합니다.


In [ ]:
PREVIEW_COUNT = 3
BASELINE_JSON = Path('/content/drive/MyDrive/bm25_only_comparison_30.json')

with QUESTION_JSON.open(encoding='utf-8-sig') as file:
    question_records = json.load(file)

# 21번 평가셋과 같이 앞의 세 사업을 선택합니다.
selected_projects = list(dict.fromkeys(record['project'] for record in question_records))[:3]
preview_records = [
    next(r for r in question_records if r['project'] == project_name)
    for project_name in selected_projects
]
for record in question_records:
    if len(preview_records) >= PREVIEW_COUNT:
        break
    if record['project'] in selected_projects and record not in preview_records:
        preview_records.append(record)
preview_records = preview_records[:PREVIEW_COUNT]
if len(preview_records) < PREVIEW_COUNT:
    raise ValueError(f'질문 {PREVIEW_COUNT}개가 필요합니다.')
preview_questions = [record['question'] for record in preview_records]

# 모델을 다운로드하기 전에 먼저 단독 검색을 실행합니다.
preview_searches = []
for record in preview_records:
    project, found = search_kiwi_bm25(record['question'])
    preview_searches.append((project, found))

baseline_by_id = {}
if BASELINE_JSON.is_file():
    baseline = json.loads(BASELINE_JSON.read_text(encoding='utf-8'))
    if baseline.get('comparison_type') != 'bm25_only':
        raise ValueError('21번 단독 검색 결과 파일이 아닙니다.')

    # 21번 저장 파일과 문서/청크 내용이 동일한지 확인합니다.
    corpus = {}
    for project_name in selected_projects:
        example = next(r for r in question_records if r['project'] == project_name)
        project = identify_project(example['question'])
        for doc in project_chunks_by_key[project]:
            corpus[str(doc.metadata['chunk_id'])] = {
                'text': doc.page_content,
                'agency': project[0],
                'project': project[1],
            }
    corpus_hash = hashlib.sha256(
        json.dumps(corpus, ensure_ascii=False, sort_keys=True).encode('utf-8')
    ).hexdigest()
    if corpus_hash != baseline.get('corpus_sha256'):
        raise ValueError('21번과 문서 또는 청크가 다릅니다. 21번을 같은 자료로 다시 실행하세요.')
    baseline_by_id = {item['id']: item for item in baseline['questions']}

for record, (project, found) in zip(preview_records, preview_searches):
    ids = [str(doc.metadata['chunk_id']) for doc in found]
    print(f"\n[{record['id']}] {record['question']}")
    print('Kiwi BM25 단독 검색 상위 20개:', ids)
    if baseline_by_id:
        old = baseline_by_id.get(record['id'])
        if old is None or old['question'] != record['question']:
            raise ValueError(f"21번 결과에 같은 문항이 없습니다: {record['id']}")
        if old['ranked_chunk_ids']['kiwi'] != ids:
            raise ValueError(f"21번과 Kiwi 검색 순위가 다릅니다: {record['id']}")
        print('21번 저장 결과와 검색 순위 일치: 예')
    else:
        print('21번 저장 결과 파일 없음: 이 노트북의 단독 검색과 아래 Qwen 답변을 비교합니다.')


## 4. Qwen3.8 양자화 모델 준비

llama.cpp를 T4용으로 빌드하고 GGUF 파일을 다운로드합니다. 모델 파일은 약 12.7GB입니다. Colab 세션 저장 공간도 확인하세요.


In [ ]:
import subprocess
from huggingface_hub import hf_hub_download

LLAMA_DIR = Path('/content/llama.cpp')
if not LLAMA_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp.git', str(LLAMA_DIR)], check=True)
subprocess.run([
    'cmake', '-S', str(LLAMA_DIR), '-B', str(LLAMA_DIR / 'build'),
    '-DGGML_CUDA=ON', '-DCMAKE_CUDA_ARCHITECTURES=75',
    '-DLLAMA_BUILD_TESTS=OFF', '-DCMAKE_BUILD_TYPE=Release',
], check=True)
subprocess.run([
    'cmake', '--build', str(LLAMA_DIR / 'build'), '--config', 'Release',
    '-j', '2', '--target', 'llama-server',
], check=True)

MODEL_REPO = 'bartowski/Qwen3.8-27B-GGUF'
MODEL_FILE = 'Qwen3.8-27B-Q3_K_S.gguf'
model_path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE)
print('양자화 모델 준비:', MODEL_FILE)


## 5. 로컬 생성 서버 시작

llama.cpp가 GPU 메모리에 맞춰 모델을 올립니다. 답변만 출력하도록 생각 모드는 끕니다. 첫 시작에는 모델을 읽느라 시간이 걸릴 수 있습니다.


In [ ]:
import time
import requests
from openai import OpenAI

server_url = 'http://127.0.0.1:8080'
server_log = open('/content/qwen38_server.log', 'w')
server = subprocess.Popen([
    str(LLAMA_DIR / 'build/bin/llama-server'),
    '-m', str(model_path),
    '--host', '127.0.0.1', '--port', '8080',
    '--gpu-layers', 'auto', '--fit', 'on',
    '-c', '4096', '--parallel', '1',
    '--reasoning', 'off',
    '--chat-template-kwargs', '{"enable_thinking": false}',
], stdout=server_log, stderr=subprocess.STDOUT)

for _ in range(180):
    if server.poll() is not None:
        raise RuntimeError('생성 서버가 중단됐습니다. /content/qwen38_server.log를 확인하세요.')
    try:
        if requests.get(server_url + '/health', timeout=2).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise TimeoutError('생성 서버 시작 시간이 초과됐습니다. /content/qwen38_server.log를 확인하세요.')

client = OpenAI(base_url=server_url + '/v1', api_key='local', timeout=300)
MODEL_NAME = client.models.list().data[0].id
print('생성 서버 준비:', MODEL_NAME)


## 6. Qwen 답변 만들기

위에서 검색한 상위 20개 중 앞의 4개를 Qwen에 전달합니다.


In [ ]:
def answer_question(question, search_result=None):
    # 미리 검색한 결과를 전달하면 Qwen에 같은 검색 결과를 그대로 사용합니다.
    project, found = search_result if search_result is not None else search_kiwi_bm25(question)
    context_docs = found[:CONTEXT_TOP_K]
    context = '\n\n'.join(
        f'[{i}] 파일: {doc.metadata["파일명"]} / 청크: {doc.metadata["chunk_id"]}\n{doc.page_content}'
        for i, doc in enumerate(context_docs, 1)
    )
    messages = [
        {'role': 'system', 'content': (
            '너는 RFP 문서 질문 답변 도우미다. 제공된 문서 내용만 근거로 한국어로 답한다. '
            '모르는 내용은 추측하지 말고 확인할 수 없다고 말한다. '
            '중요한 사실 뒤에는 근거 번호 [1]처럼 붙인다.'
        )},
        {'role': 'user', 'content': f'<문서>\n{context}\n</문서>\n\n<질문>\n{question}\n</질문>'},
    ]
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        max_tokens=768,
        temperature=0.7,
        top_p=0.8,
        extra_body={'top_k': 20, 'chat_template_kwargs': {'enable_thinking': False}},
    )
    return {
        'question': question,
        'project': project[1],
        'answer': (response.choices[0].message.content or '').strip(),
        'search_chunk_ids': [str(doc.metadata['chunk_id']) for doc in found],
        'context_chunk_ids': [str(doc.metadata['chunk_id']) for doc in context_docs],
    }


## 7. 같은 질문에서 검색 결과와 답변 보기

21번과 같은 Kiwi 검색 상위 20개, 실제로 Qwen에 전달한 4개, Qwen 답변, 평가셋 참고 답변을 함께 출력합니다.


In [ ]:
preview_results = []
for number, (record, search_result) in enumerate(zip(preview_records, preview_searches), 1):
    result = answer_question(record['question'], search_result=search_result)
    preview_results.append(result)
    print(f"\n========== {number}. {record['id']} ==========")
    print('질문:', record['question'])
    print('Kiwi BM25 단독 검색 상위 20개:', result['search_chunk_ids'])
    print('Qwen에 전달한 앞의 4개:', result['context_chunk_ids'])
    for rank, doc in enumerate(search_result[1][:CONTEXT_TOP_K], 1):
        snippet = doc.page_content[:160].replace('\n', ' ')
        print(f'  [{rank}] {doc.metadata["파일명"]} / 청크 {doc.metadata["chunk_id"]}: {snippet}...')
    print('Qwen 답변:', result['answer'])
    print('평가셋 참고 답변:', record['reference'])


## 8. LangSmith 기록 (지금은 실행 안 함)

나중에 기록이 필요할 때만 아래의 `RUN_LANGSMITH`를 `True`로 바꾸세요. 기본 미리보기에는 API 키가 필요하지 않습니다. 켜면 같은 3문항을 다시 생성해 LangSmith에 질문·답변·청크 번호를 기록합니다. 채점은 하지 않습니다.


In [ ]:
RUN_LANGSMITH = False

if RUN_LANGSMITH:
    import os
    from google.colab import userdata
    from langsmith import traceable

    api_key = userdata.get('LANGSMITH_API_KEY')
    if not api_key:
        raise ValueError('Colab 비밀에 LANGSMITH_API_KEY를 등록하세요.')
    os.environ['LANGSMITH_API_KEY'] = api_key
    os.environ['LANGSMITH_TRACING'] = 'true'
    os.environ['LANGSMITH_PROJECT'] = 'rfp-kiwi-bm25-qwen38-preview'

    @traceable(name='kiwi_bm25_qwen38_answer')
    def traced_answer(question):
        return answer_question(question)

    for question in preview_questions:
        traced_answer(question)
    print('LangSmith 기록 완료:', len(preview_questions), '문항')
else:
    print('LangSmith 실행 안 함')
